In [1]:
import pandas as pd
import numpy as np

# 1. Load dataset (path disesuaikan karena notebook berada di dalam folder 'notebooks')
movies = pd.read_csv('../dataset/tmdb_5000_movies.csv')
credits = pd.read_csv('../dataset/tmdb_5000_credits.csv')

# 2. Gabungkan dataset berdasarkan judul film
movies = movies.merge(credits, on='title')

# Tampilkan kolom apa saja yang ada di dataset baru kita
print("Data berhasil digabungkan!")
print("Jumlah baris dan kolom:", movies.shape)
movies.head(3)

Data berhasil digabungkan!
Jumlah baris dan kolom: (4809, 23)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...",...,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,206647,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."


In [2]:
import ast

# 1. Pilih kolom yang relevan saja untuk sistem rekomendasi
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords']]

# 2. Fungsi pembantu untuk mengekstrak nama genre/keyword dari teks JSON mentah
def convert(text):
    L = []
    # ast.literal_eval digunakan untuk mengubah string representasi list/dict menjadi list/dict asli Python
    for i in ast.literal_eval(text):
        L.append(i['name']) 
    return L

# Hapus data yang barisnya kosong (null) agar tidak menyebabkan error
movies.dropna(inplace=True)

# 3. Bersihkan kolom genres dan keywords menggunakan fungsi convert tadi
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

# Tampilkan hasil data setelah dibersihkan
movies.head(3)

,movie_id,title,overview,genres,keywords
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi..."


In [3]:
# 1. Bersihkan spasi antar kata pada genres dan keywords (contoh: "Science Fiction" -> "sciencefiction")
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ","") for i in x])

# 2. Ubah overview dari teks biasa (string) menjadi list kata agar bisa digabungkan dengan list lainnya
movies['overview'] = movies['overview'].apply(lambda x: x.split())

# 3. Gabungkan ketiga kolom tersebut menjadi satu kolom bernama 'tags'
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords']

# 4. Buat DataFrame baru bernama 'new_df' yang hanya berisi kolom penting saja
new_df = movies[['movie_id', 'title', 'tags']]

# 5. Gabungkan kembali list kata di dalam 'tags' menjadi satu kalimat string utuh
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

# 6. Ubah semua teks di kolom tags menjadi huruf kecil (lowercase)
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

# Tampilkan hasil akhir tabel data rekomendasi kita
new_df.head(3)

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...


In [5]:
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
  Using cached scikit_learn-1.9.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.18.0-cp313-cp313-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.9.0-cp313-cp313-win_amd64.whl (8.2 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.18.0-cp313-cp313-win_amd64.whl (36.6 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   -------


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Ekstrak teks menjadi vektor angka (max 5000 kata unik, buang stop words)
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()

# 2. Hitung tingkat kemiripan antar film (Cosine Similarity)
similarity = cosine_similarity(vectors)

# 3. Buat fungsi rekomendasi sederhana
def recommend(movie_title):
    # Cari index film berdasarkan judul
    try:
        movie_index = new_df[new_df['title'] == movie_title].index[0]
        # Ambil jarak kemiripan film tersebut dengan semua film lainnya
        distances = similarity[movie_index]
        # Urutkan dari yang paling mirip (indeks 1 sampai 6 karena indeks 0 adalah film itu sendiri)
        movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
        
        print(f"Rekomendasi film setelah menonton '{movie_title}':")
        for i in movies_list:
            print("- ", new_df.iloc[i[0]].title)
    except IndexError:
        print(f"Film '{movie_title}' tidak ditemukan di dalam dataset.")

# 4. Tes jalankan fungsi rekomendasi (contoh memakai film Avatar)
recommend('Avatar')

Rekomendasi film setelah menonton 'Avatar':
-  Titan A.E.
-  Small Soldiers
-  Independence Day
-  Aliens vs Predator: Requiem
-  Battle: Los Angeles


In [7]:
import pickle
import os

# Memastikan folder 'models' ada di dalam folder 'AI'
# Jika struktur foldernya adalah AI/notebooks, kita naik satu tingkat ke folder 'AI' lalu buat folder 'models'
model_dir = os.path.join('..', 'models')
os.makedirs(model_dir, exist_ok=True)

# 1. Simpan DataFrame film (kita ubah jadi dictionary agar ukuran filenya lebih ringan saat di-load)
pickle.dump(new_df.to_dict(), open(os.path.join(model_dir, 'movie_dict.pkl'), 'wb'))

# 2. Simpan matriks similarity (tingkat kemiripan film)
pickle.dump(similarity, open(os.path.join(model_dir, 'similarity.pkl'), 'wb'))

print("Mantap! File 'movie_dict.pkl' dan 'similarity.pkl' berhasil disimpan di folder models!")

Mantap! File 'movie_dict.pkl' dan 'similarity.pkl' berhasil disimpan di folder models!
